In [4]:
!pip install "transformers[torch]"

In [5]:
import pandas as pd
from  transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [6]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [8]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [9]:
train_data["dialogue"][1]

'Olivia: Who are you voting for in this election? \r\nOliver: Liberals as always.\r\nOlivia: Me too!!\r\nOliver: Great'

In [10]:
train_data.sample(5)

,id,dialogue,summary
4482,13680930,Harry: hi\r\nHenry: hello\r\nHarry: she agreed...,Hermiona has finally agreed to go on a date wi...
6678,13680998,Charlie: I love that city...\r\nKate: ?\r\nCha...,Today some idiot called the municipal police a...
6579,13821199,Carl: I'm waking you up at 7 tomorrow!\r\nRoge...,Carl's waking Roger and Brad up at 7 tomorrow.
9519,13728092,"Timmy: Still, he's great company. Has loads of...","Gemma is about to invite guests, and she's con..."
8236,13820401,Catherine: i hate this bitch\r\nCatherine: why...,"Apparently, Cassidy speaks about Catherine in ..."


In [11]:
train_data.shape

(14732, 3)

In [12]:
val_data.shape

(818, 3)

In [13]:
# random sampling 
# we take 4000 sample out of 14732 for training and 500 sample for validation

train_data = train_data.sample(n = 4000, random_state = 42).reset_index(drop = True)
val_data = train_data.sample(n = 500, random_state = 42).reset_index(drop = True)

In [14]:
train_data.shape
val_data.shape

(500, 3)

# Data Preprocessing

In [15]:
# we use re -- regular expression module for cleaning our data

import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text)  # html tags <p> <h1>
    text = text.strip().lower()  # remove extra spaces from start n end and convert all text into lowercase
    return text

In [16]:
# training data
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

# validation data 
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)



In [17]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize

In [18]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")   # t-5 is T5 transformer

In [19]:
# raw data ==> tokenize inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs
    

In [20]:
train_data = train_data.apply(tokenize, axis = 1 ).tolist()
val_data = val_data.apply(tokenize, axis = 1).tolist()

In [21]:
train_data[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [22]:
# inputs ids ==> dialogue ==> token ids

# 1 => End of sequence, 0 ==> padding

# attention mask
# labels -- target ==> summary tokens

In [23]:
len(train_data[0]["input_ids"])

512

In [24]:
len(train_data[2000]["attention_mask"])

len(train_data[2000]["labels"])

150

In [25]:
val_data[0]

{'input_ids': [836, 6961, 10, 3, 13133, 1240, 226, 6, 103, 25, 341, 3884, 22269, 58, 1240, 226, 10, 7102, 3158, 307, 97, 150, 1616, 3, 10, 61, 17945, 6, 3, 23, 103, 6, 125, 31, 7, 95, 58, 836, 6961, 10, 320, 6, 82, 7930, 19, 479, 21, 841, 113, 228, 3884, 376, 6, 78, 3, 23, 816, 81, 25, 5, 1240, 226, 10, 417, 589, 55, 125, 593, 19, 3, 88, 44, 230, 58, 836, 6961, 10, 3, 107, 51, 233, 614, 12, 817, 6, 43, 29, 31, 17, 1380, 3, 117, 61, 3, 23, 317, 3, 88, 131, 2746, 12, 36, 6720, 295, 6, 78, 3, 88, 1077, 523, 128, 12, 43, 9029, 28, 5, 1240, 226, 10, 150, 682, 6, 8, 589, 19, 3, 23, 163, 43, 1911, 1135, 7, 11, 62, 26, 1496, 1135, 7, 326, 233, 836, 6961, 10, 3, 32, 107, 82, 6, 24, 31, 7, 626, 55, 3, 88, 31, 7, 882, 3, 9, 3164, 4024, 6, 68, 3, 88, 243, 62, 26, 1496, 1135, 2993, 1399, 5, 1240, 226, 10, 248, 258, 55, 2049, 21, 48, 55, 3, 10, 61, 836, 6961, 10, 4244, 1024, 1024, 6, 150, 682, 55, 78, 6, 54, 3, 23, 1499, 376, 39, 381, 58, 1240, 226, 10, 13, 503, 3, 10, 61, 836, 6961, 10, 3, 88, 31, 

In [26]:
len(val_data[0]["input_ids"])

len(val_data[400]["attention_mask"])

512

In [27]:
type(val_data)

list

# Working with our Model

In [28]:
# NLP ==> generation task 
# output generate on basis of dialogue so its conditional generation

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [29]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [30]:
# Training Arguments 

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
    # 0 ==> lr default
    # transformer learning rate start with 0 and in 500 steps it reaches to default value 5e-05
)

In [31]:
# Trainer
# The Trainer class provides an API for feature-complete training in PyTorch, 
# and it supports distributed training on multiple GPUs/TPUs

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_data,
    eval_dataset = val_data
)


In [28]:
# train the model

trainer.train()

C:\Users\RanjeetKumar\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.620698,0.367770
2,0.397371,0.338281
3,0.374184,0.327115
4,0.363053,0.318726
5,0.356382,0.314183
6,0.351242,0.312736


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\RanjeetKumar\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

C:\Users\RanjeetKumar\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\RanjeetKumar\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9104884134928385, metrics={'train_runtime': 97441.225, 'train_samples_per_second': 0.246, 'train_steps_per_second': 0.031, 'total_flos': 3248203235328000.0, 'train_loss': 0.9104884134928385, 'epoch': 6.0})

In [ ]:
# model load ==> fine-tune ==> save the model

In [30]:
# save the model and tokenizers

model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [32]:
# use our save model

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core logic for summarization

In [37]:
def summarize_dialogue(dialogue):

    # clean data
    dialogue = clean_data(dialogue)

    # tokenize
    inputs = tokenizer(
        dialogue, 
        padding = "max_length", 
        max_length = 512, 
        truncation = True,
        return_tensors = "pt"  # convert tokens in tensors, pt--> pytorch tensors
    ).to(device)

    #  generate the summary ==> token ids
    model.to(device)
    target = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,   # In Ai there is one algorithm which is beam search, which will generate 4 outputs, and give best out of it
        early_stopping = True
        
    )

    # token ids convert to summary 
    # decoded our output 
    summary = tokenizer.decode(target[0], skip_special_tokens = True)   # EOS, seperator e.g <html> space 
    return summary

In [38]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary:", summary)

Summary: report: companies investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
